# Deploy camera-trap-image-classifier to SageMaker Serverless Endpoint

This notebook deploys the camera-trap-image-classifier model to a SageMaker serverless endpoint. It is intended to be run in a SageMaker Notebook instance on the `conda_pytorch_p10` kernel.

The code is adapted from 
https://github.com/aws-samples/amazon-sagemaker-endpoint-deployment-of-fastai-model-with-torchserve

## Setup

### Import Dependencies

In [ ]:
%matplotlib inline

import boto3
import time
import json
import base64
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import sagemaker
from io import BytesIO

### Initialize AWS Session

In [ ]:
sess = boto3.Session()
sm = sess.client('sagemaker')
region = sess.region_name
account = boto3.client('sts').get_caller_identity().get('Account')

### Get IAM Role

**Note**: Ensure the IAM role has:
- `AmazonS3FullAccess`
- `AmazonSageMakerFullAccess`

In [ ]:
role = sagemaker.get_execution_role()
print(f"Using role: {role}")

## Create ECR Registry

In [ ]:
# Create ECR repository if it doesn't exist
registry_name = "torchserve-camera-trap-vehicle-classifier-sagemaker"
ecr = boto3.client('ecr')

try:
    ecr.create_repository(repositoryName=registry_name)
except ecr.exceptions.RepositoryAlreadyExistsException:
    pass

image = f"{account}.dkr.ecr.{region}.amazonaws.com/{registry_name}:latest"

## Build and Push Container

Create a compressed `*.tar.gz` file from the `*.mar` file per requirement of Amazon SageMaker and upload the model to your Amazon S3 bucket.

**Skip this step if the registry is already made and the custom latest pytorch container is already pushed since this step takes a couple of minutes**

In [ ]:
model_prefix = "camera-trap-vehicle-classifier"
model_uri = f's3://animl-model-zoo/{model_prefix}/exported-model/{model_prefix}.mar'
sagemaker_session = sagemaker.Session(boto_session=sess)
bucket_name = sagemaker_session.default_bucket()
prefix = 'torchserve'
prod_model_uri = f"s3://{bucket_name}/{prefix}/models/"

In [ ]:
!aws s3 cp {model_uri} ./
!tar cvfz {model_prefix}.tar.gz {model_prefix}.mar
!aws s3 cp {model_prefix}.tar.gz {prod_model_uri}

In [ ]:
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account}.dkr.ecr.{region}.amazonaws.com
!docker build -t {registry_name} .
!docker tag {registry_name} {image}
!docker push {image}

## Create SageMaker Model

In [ ]:
model_prefix = "camera-trap-vehicle-classifier" if model_prefix is None else model_prefix

# Check if model already exists
model_data = f"{prod_model_uri}{model_prefix}.tar.gz"
model_already_created = False
for model_def in sm.list_models()['Models']:
    if model_prefix == model_def['ModelName']:
        create_model_response = model_def
        model_already_created = True

# Create model if it doesn't exist
if not model_already_created:
    container = {"Image": image, "ModelDataUrl": model_data}

    if not model_already_created:
        create_model_response = sm.create_model(
            ModelName=model_prefix, ExecutionRoleArn=role, PrimaryContainer=container
        )

    print(create_model_response["ModelArn"])

print(f"Model ARN: {create_model_response['ModelArn']}")

## Create Endpoint Configurations

In [ ]:
# Create realtime and batch endpoint configuration
# for batch endpoints, use concurrency of 80, for real-time endpoints, use 20
# https://github.com/tnc-ca-geo/animl-api/issues/101

batch_endpoint_config_name = f"{model_prefix}-config-concurrency-80"
batch_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=batch_endpoint_config_name,
    ProductionVariants=[
        {
            "ModelName": model_prefix,
            "VariantName": "AllTraffic",
            "ServerlessConfig": {
                "MemorySizeInMB": 4096,  # 4GB memory
                "MaxConcurrency": 80     # Maximum concurrent invocations
            }
        }
    ]
)
print(f"Endpoint Config ARN: {batch_endpoint_config_response['EndpointConfigArn']}")

In [ ]:
# Enable realtime endpoint config creation if not needed
create_realtime_endpoint_config = False

if create_realtime_endpoint_config:
    realtime_endpoint_config_name = f"{model_prefix}-config-concurrency-20"
    realtime_endpoint_config_response = sm.create_endpoint_config(
        EndpointConfigName=realtime_endpoint_config_name,
        ProductionVariants=[
            {
                "ModelName": model_prefix,
                "VariantName": "AllTraffic",
                "ServerlessConfig": {
                    "MemorySizeInMB": 4096,  # 4GB memory
                    "MaxConcurrency": 20     # Maximum concurrent invocations
                }
            }
        ]
    )
    print(f"Endpoint Config ARN: {realtime_endpoint_config_response['EndpointConfigArn']}")

## Create and Deploy Endpoints

In [ ]:
# Create batch endpoint

batch_endpoint_name = f"{model_prefix}-concurrency-80"
create_batch_endpoint_response = sm.create_endpoint(
    EndpointName=batch_endpoint_name,
    EndpointConfigName=batch_endpoint_config_name
)

print(f"Batch Endpoint ARN: {create_batch_endpoint_response['EndpointArn']}")

# Wait for batch endpoint creation
batch_resp = sm.describe_endpoint(EndpointName=batch_endpoint_name)
batch_status = batch_resp['EndpointStatus']
print(f"Status: {batch_status}")

while batch_status == 'Creating':
    time.sleep(60)
    batch_resp = sm.describe_endpoint(EndpointName=batch_endpoint_name)
    batch_status = batch_resp['EndpointStatus']
    print(f"Status: {batch_status}")
    if batch_status == 'Failed':
        failure_reason = batch_resp.get('FailureReason', 'No failure reason provided')
        print(f"Batch endpoint deployment failed: {failure_reason}")
        break
print(f"Batch Arn: {batch_resp['EndpointArn']}")
print(f"Batch endpoint final status: {batch_status}")
if batch_status == 'Failed':
    batch_log_group = f"/aws/sagemaker/Endpoints/{batch_endpoint_config_name}"
    try:
        log_streams = logs.describe_log_streams(logGroupName=batch_log_group)
        for stream in log_streams['logStreams']:
            print(f"\nLog stream: {stream['logStreamName']}")
            batch_events = logs.get_log_events(logGroupName=batch_log_group, logStreamName=stream['logStreamName'])
            for event in batch_events['events']:
                print(event['message'])
    except Exception as e:
        print(f"Error fetching logs: {str(e)}")

In [ ]:
# Create realtime endpoint (if needed)

if create_realtime_endpoint_config:
    realtime_endpoint_name = f"{model_prefix}-concurrency-20"
    create_realtime_endpoint_response = sm.create_endpoint(
        EndpointName=realtime_endpoint_name,
        EndpointConfigName=realtime_endpoint_config_name
    )
    
    print(f"Endpoint ARN: {create_realtime_endpoint_response['EndpointArn']}")
    
    # Wait for endpoint creation
    resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
    realtime_status = resp['EndpointStatus']
    print(f"Status: {realtime_status}")
    
    while realtime_status == 'Creating':
        time.sleep(60)
        resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
        realtime_status = resp['EndpointStatus']
        print(f"Status: {realtime_status}")
        if realtime_status == 'Failed':
            realtime_failure_reason = resp.get('FailureReason', 'No failure reason provided')
            print(f"Realtime endpoint deployment failed: {realtime_failure_reason}")
            break
    
    # Get CloudWatch logs for the endpoint
    logs = boto3.client('logs')
    
    print(f"Realtime Arn: {resp['EndpointArn']}")
    print(f"Realtime endpoint final status: {realtime_status}")
    if realtime_status == 'Failed':
        realtime_log_group = f"/aws/sagemaker/Endpoints/{realtime_endpoint_name}"
        try:
            log_streams = logs.describe_log_streams(logGroupName=realtime_log_group)
            for stream in log_streams['logStreams']:
                print(f"\nLog stream: {stream['logStreamName']}")
                realtime_events = logs.get_log_events(logGroupName=realtime_log_group, logStreamName=stream['logStreamName'])
                for event in realtime_events['events']:
                    print(event['message'])
        except Exception as e:
            print(f"Error fetching logs: {str(e)}")

## Test the Endpoint

In [ ]:
# Load a test image from the test-data directory
endpoint_name = batch_endpoint_name
test_image = Image.open("tests/test-data/mountain-biker-test.jpg")
display(test_image)

# Convert image to base64
buffered = BytesIO()
test_image.save(buffered, format="JPEG")
img_str = base64.b64encode(buffered.getvalue()).decode()

# Prepare payload
payload = {
    "image": img_str,
    "bbox": [0.19789853692054749,0,0.9022471904754639,0.5614370703697205]
}

# Invoke endpoint
client = boto3.client('runtime.sagemaker')
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="multipart/form-data",
    Body=json.dumps(payload)
)

# Parse results
result = json.loads(response['Body'].read().decode())
print("\nPrediction Results:")
print(json.dumps(result, indent=2))

## Cleanup Resources

**Note**: Only run this cell when you want to delete the endpoint and associated resources.

In [ ]:
# Uncomment to cleanup
# client = boto3.client('sagemaker')
# client.delete_endpoint(EndpointName=realtime_endpoint_name)
# client.delete_endpoint_config(EndpointConfigName=realtime_endpoint_config_name)
# client.delete_model(ModelName=model_prefix)